# Intrusion Detection on CICIDS2017 — Reproduction & Critique

**Course:** Data Science Methods in Cyber Security (Dr. Uri Itai, University of Haifa)  
**Student:** Eyal Steinmetz  
**Deadline:** Friday, 10 July 2026, 23:59

**Mission:** Reproduce Rodríguez et al. (2022, *Sensors* MDPI), who claim a Random Forest reaches
**F1 > 0.999** on CICIDS2017, then demonstrate that the claim is an artifact of three documented
methodological flaws — duplicate contamination, temporal leakage, and aggregate-only metrics —
using Engelen et al. (2021) as peer-reviewed counter-evidence.

| Notebook section | Report section | Purpose |
|---|---|---|
| 0. Setup | front matter | Imports, seed, paths, figure helper |
| **1. Data Loading & Cleaning** | §3 Data | Load 8 day-files, tag `day`, strip columns, quantify quality issues |
| **2. EDA** | §4 EDA | Class balance, temporal structure, Spearman correlation |
| **3. Feature Engineering** | §5 | Encoding, log-transform, correlation pruning (78 → 40 features) |
| 4. Model Training | §6 | Strategy A (random) vs Strategy B (dedup + temporal) |
| 5. Evaluation | §6/§7 | Metric suite, confusion matrices, ROC/PR |
| 6. Error Analysis & Critique | §2 | Three-flaw framework, per-class failure, verdict |
| 7. Executive Summary | §1/§8 | Problem → method → findings → verdict |

> Sections **0–3** are implemented (Setup, Data Loading & Cleaning, EDA, Feature Engineering) — Phases 2–4 in the project plan. Sections 4–7 (modelling, evaluation, critique) follow.

## Section 0 — Setup & Imports

We fix a single global random seed (`RANDOM_STATE = 42`) used by every split, model, and SMOTE
call for reproducibility, define the project paths, and provide a `save_fig()` helper so every
figure lands in `figures/` at a consistent resolution. The notebook lives in `notebooks/`, so
all paths are relative to that directory.

In [1]:
# Standard library
import warnings
from pathlib import Path

# Third-party (data + visualization)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Reproducibility — one seed used everywhere
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths (notebook runs from notebooks/)
DATA_DIR = Path("../data/MachineLearningCVE")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Warnings policy: silence only FutureWarning noise from pandas/sklearn version churn.
# We do NOT silence SettingWithCopyWarning — that one signals real bugs and must surface.
warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid")


def save_fig(fig, name, tight=True):
    """Save a figure to figures/ with consistent settings, then close it."""
    path = FIGURES_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight" if tight else None)
    plt.close(fig)
    print(f"Saved: {path}")


print("Imports OK")
print(f"RANDOM_STATE = {RANDOM_STATE}")
print(f"DATA_DIR     = {DATA_DIR.resolve()}")
print(f"FIGURES_DIR  = {FIGURES_DIR.resolve()}")

Imports OK
RANDOM_STATE = 42
DATA_DIR     = C:\Users\אייל. ש\Documents\HUP\cyber\project\data\MachineLearningCVE
FIGURES_DIR  = C:\Users\אייל. ש\Documents\HUP\cyber\project\figures


## Section 1 — Data Loading & Cleaning

We load CICIDS2017 from its pre-extracted CSV flow features (`MachineLearningCVE/`). The published
release ships **8** CSV files, not 5: Thursday is split into *WebAttacks* and *Infiltration*
captures, and Friday into *Morning*, *PortScan*, and *DDoS* captures. Because the ML CSVs carry no
timestamp column, we derive the weekday — needed later for the temporal split (Flaw 2) — directly
from each filename.

### 1.1 — Load all day-files and concatenate

Each CSV is one capture. We tag every row with its source weekday (`day_name`) and an ordinal
`day` index (Mon=0 … Fri=4) *before* concatenating, then strip whitespace from column names
immediately — CICFlowMeter emits leading/trailing spaces in headers (Engelen et al., 2021).

In [2]:
# Ordinal weekday encoding for the temporal split (Mon=0 .. Fri=4)
WEEKDAY_ORDER = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3, "Friday": 4}


def weekday_from_filename(filename: str):
    """Derive (weekday_name, ordinal) from a CICIDS2017 CSV filename.

    The 8-file release prefixes every file with the weekday, e.g.
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv'. Case-insensitive
    because the Wednesday file uses a lowercase 'workingHours'.
    """
    low = filename.lower()
    for name, idx in WEEKDAY_ORDER.items():
        if low.startswith(name.lower()):
            return name, idx
    raise ValueError(f"Cannot derive weekday from filename: {filename}")


csv_paths = sorted(DATA_DIR.glob("*.pcap_ISCX.csv"))
assert csv_paths, f"No CSVs found in {DATA_DIR.resolve()} — place the MachineLearningCVE files there."

frames = []
for path in csv_paths:
    day_name, day_idx = weekday_from_filename(path.name)
    part = pd.read_csv(path, low_memory=False)
    part["day"] = day_idx
    part["day_name"] = day_name
    frames.append(part)
    print(f"  {path.name:<55} -> {day_name:<9} ({len(part):>8,} rows)")

df = pd.concat(frames, ignore_index=True)
df.columns = df.columns.str.strip()  # strip once: CICFlowMeter headers carry whitespace

print(f"\nLoaded {len(df):,} rows x {len(df.columns)} columns from {len(csv_paths)} files")
print("\nRows per day:")
print(df["day_name"].value_counts().reindex(WEEKDAY_ORDER.keys()).to_string())

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv        -> Friday    ( 225,745 rows)


  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv    -> Friday    ( 286,467 rows)


  Friday-WorkingHours-Morning.pcap_ISCX.csv               -> Friday    ( 191,033 rows)


  Monday-WorkingHours.pcap_ISCX.csv                       -> Monday    ( 529,918 rows)


  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv -> Thursday  ( 288,602 rows)


  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  -> Thursday  ( 170,366 rows)


  Tuesday-WorkingHours.pcap_ISCX.csv                      -> Tuesday   ( 445,909 rows)


  Wednesday-workingHours.pcap_ISCX.csv                    -> Wednesday ( 692,703 rows)



Loaded 2,830,743 rows x 81 columns from 8 files

Rows per day:
day_name
Monday       529918
Tuesday      445909
Wednesday    692703
Thursday     458968
Friday       703245


### 1.2 — First inspection: dtypes, columns, labels

We confirm the column names are clean, locate the only expected non-numeric feature columns
(`Label` plus our two tag columns), flag the known redundant `Fwd Header Length.1` duplicate
column, and print the full label distribution. The label counts must show the severe imbalance
(~80% BENIGN) and the ultra-rare attack classes (Heartbleed, SQL Injection, Infiltration) that
later expose Flaw 3.

In [3]:
# Non-numeric columns (expect only Label + our two tag columns)
non_numeric = df.select_dtypes(exclude=np.number).columns.tolist()
print(f"Non-numeric columns ({len(non_numeric)}): {non_numeric}")

# Known redundant column from CICFlowMeter (Engelen et al., 2021) — flag now, drop in Phase 3
dup_col = "Fwd Header Length.1"
print(f"\n'{dup_col}' present (redundant, drop later): {dup_col in df.columns}")

print(f"\nTotal feature columns (excl. Label/day/day_name): {len(df.columns) - 3}")
print("\nAll column names:")
for i, c in enumerate(df.columns):
    print(f"  [{i:>2}] {c}")

Non-numeric columns (2): ['Label', 'day_name']

'Fwd Header Length.1' present (redundant, drop later): True

Total feature columns (excl. Label/day/day_name): 78

All column names:
  [ 0] Destination Port
  [ 1] Flow Duration
  [ 2] Total Fwd Packets
  [ 3] Total Backward Packets
  [ 4] Total Length of Fwd Packets
  [ 5] Total Length of Bwd Packets
  [ 6] Fwd Packet Length Max
  [ 7] Fwd Packet Length Min
  [ 8] Fwd Packet Length Mean
  [ 9] Fwd Packet Length Std
  [10] Bwd Packet Length Max
  [11] Bwd Packet Length Min
  [12] Bwd Packet Length Mean
  [13] Bwd Packet Length Std
  [14] Flow Bytes/s
  [15] Flow Packets/s
  [16] Flow IAT Mean
  [17] Flow IAT Std
  [18] Flow IAT Max
  [19] Flow IAT Min
  [20] Fwd IAT Total
  [21] Fwd IAT Mean
  [22] Fwd IAT Std
  [23] Fwd IAT Max
  [24] Fwd IAT Min
  [25] Bwd IAT Total
  [26] Bwd IAT Mean
  [27] Bwd IAT Std
  [28] Bwd IAT Max
  [29] Bwd IAT Min
  [30] Fwd PSH Flags
  [31] Bwd PSH Flags
  [32] Fwd URG Flags
  [33] Bwd URG Flags
  [34] Fwd H

In [4]:
# Normalize label text: the raw CSVs encode the web-attack dash as a non-UTF-8 byte that
# reads back as the replacement char (e.g. "Web Attack � XSS"). Map it to a plain hyphen
# and strip whitespace so per-class reporting (Flaw 3) uses clean, stable class names.
df["Label"] = df["Label"].str.replace("�", "-", regex=False).str.replace("  ", " ").str.strip()

# Label distribution — counts and percentages
label_counts = df["Label"].value_counts()
label_pct = df["Label"].value_counts(normalize=True) * 100
label_summary = pd.DataFrame({"count": label_counts, "pct": label_pct.round(4)})
print(label_summary.to_string())

benign_pct = label_pct.get("BENIGN", 0.0)
print(f"\nBENIGN share: {benign_pct:.1f}%   |   Attack share: {100 - benign_pct:.1f}%")
print(f"Distinct labels: {df['Label'].nunique()}")

                              count      pct
Label                                       
BENIGN                      2273097  80.3004
DoS Hulk                     231073   8.1630
PortScan                     158930   5.6144
DDoS                         128027   4.5227
DoS GoldenEye                 10293   0.3636
FTP-Patator                    7938   0.2804
SSH-Patator                    5897   0.2083
DoS slowloris                  5796   0.2048
DoS Slowhttptest               5499   0.1943
Bot                            1966   0.0695
Web Attack - Brute Force       1507   0.0532
Web Attack - XSS                652   0.0230
Infiltration                     36   0.0013
Web Attack - Sql Injection       21   0.0007
Heartbleed                       11   0.0004

BENIGN share: 80.3%   |   Attack share: 19.7%
Distinct labels: 15


### 1.3 — Quality issue: Inf and NaN values

CICFlowMeter divides by flow duration to produce `Flow Bytes/s` and `Flow Packets/s`; zero-duration
flows yield `Inf` (Engelen et al., 2021). We count these, convert `±Inf` to `NaN`, report which
columns carry missing values, and drop the affected rows. This minimal cleaning is identical for
both evaluation strategies, so doing it once here introduces no leakage.

In [5]:
numeric_cols = df.select_dtypes(include=np.number).columns

# 1. Locate Inf cells once, then reuse the same mask to set them to NaN.
#    (One numeric-only scan — no second full-frame df.replace pass.)
inf_mask = np.isinf(df[numeric_cols])
inf_total = int(inf_mask.to_numpy().sum())
inf_by_col = inf_mask.sum()
inf_by_col = inf_by_col[inf_by_col > 0]
print(f"Total Inf/-Inf cells: {inf_total:,}")
print("Columns carrying Inf:")
print(inf_by_col.to_string())

df[numeric_cols] = df[numeric_cols].mask(inf_mask)  # True -> NaN, reusing the mask

# 2. NaN counts per column (after Inf conversion)
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
print(f"\nColumns with NaN (post Inf->NaN): {len(nan_cols)}")
print(nan_cols.to_string())

# 3. Drop rows with any NaN (document the count)
n_before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"\nDropped {n_before - len(df):,} rows containing NaN ({(n_before - len(df)) / n_before * 100:.3f}%)")
print(f"Remaining rows: {len(df):,}")

Total Inf/-Inf cells: 4,376
Columns carrying Inf:
Flow Bytes/s      1509
Flow Packets/s    2867



Columns with NaN (post Inf->NaN): 2
Flow Bytes/s      2867
Flow Packets/s    2867



Dropped 2,867 rows containing NaN (0.101%)
Remaining rows: 2,827,876


In [6]:
# Negative values in flow features that are physically non-negative (counts, lengths, durations).
# CICFlowMeter is known to emit spurious negatives (Engelen et al., 2021); we document, not fix, here.
feature_cols = [c for c in numeric_cols if c not in ("day",)]
neg_counts = (df[feature_cols] < 0).sum()
neg_cols = neg_counts[neg_counts > 0].sort_values(ascending=False)
print(f"Feature columns containing negative values: {len(neg_cols)}")
print(neg_cols.to_string() if len(neg_cols) else "  (none)")

Feature columns containing negative values: 13
Init_Win_bytes_backward    1439672
Init_Win_bytes_forward     1001172
Flow IAT Min                  2890
Flow Packets/s                 115
Flow Duration                  115
Flow IAT Mean                  115
Flow IAT Max                   115
Flow Bytes/s                    85
Fwd Header Length               35
Fwd Header Length.1             35
min_seg_size_forward            35
Bwd Header Length               22
Fwd IAT Min                     17


### 1.4 — Quality issue: duplicate rows (Flaw 1)

CICIDS2017 contains a large number of **exact duplicate flow records** (Engelen et al., 2021). Under
a random train/test split these duplicates are scattered across both sets, letting the model
"memorize" test points it has already seen in training — inflating every metric. This is **Flaw 1**
of our critique.

We **quantify** the duplicates here and persist the count to `figures/dedup_stats.txt`, but we do
**not** remove them yet. Deduplication is deferred to **Strategy B** so we can measure the exact
F1 inflation that the duplicates cause (run A0 with duplicates vs A1 without).

In [7]:
# Duplicates over the feature + label columns (ignore the tag columns we added).
dedup_subset = [c for c in df.columns if c not in ("day", "day_name")]
n_dupes = int(df.duplicated(subset=dedup_subset).sum())
pct_dupes = n_dupes / len(df) * 100
print(f"FLAW 1 - exact duplicate rows: {n_dupes:,} ({pct_dupes:.2f}% of data)")
print("Duplicates are RETAINED here on purpose; removal is deferred to Strategy B (A0 vs A1).")

# Persist for the report (cited verbatim in the critique)
stats_path = FIGURES_DIR / "dedup_stats.txt"
stats_path.write_text(
    f"rows_total={len(df)}\n"
    f"duplicate_rows={n_dupes}\n"
    f"duplicate_pct={pct_dupes:.4f}\n",
    encoding="utf-8",
)
print(f"Saved: {stats_path}")

FLAW 1 - exact duplicate rows: 307,078 (10.86% of data)
Duplicates are RETAINED here on purpose; removal is deferred to Strategy B (A0 vs A1).
Saved: ..\figures\dedup_stats.txt


### 1.5 — Zero-variance columns, artifacts, memory, and dtype downcast

We list **zero-variance** feature columns (constant for every row) — they carry no signal and are
dropped in Phase 3 — count the `Flow Duration == 0` CICFlowMeter artifacts, and report the
DataFrame's memory footprint. We then **downcast** the numeric columns to shrink the ~2 GB
footprint (ADR-10 / TODO P2).

**Ordering matters:** the downcast runs *after* every quality count above (duplicates, negatives,
zero-variance). Casting `float64 → float32` could collapse two rows that differ only beyond
float32 precision into an exact duplicate, which would inflate the Flaw 1 number — so all reported
quality statistics are computed at full precision first. We let pandas pick the smallest *safe*
type per column (`pd.to_numeric(downcast=...)`); this keeps `Destination Port` (max 65535) in
`int32` rather than overflowing `int16`, and we assert min/max are preserved.

In [8]:
# Zero-variance feature columns (constant) — candidates to drop in Phase 3
feature_only = df.drop(columns=["Label", "day", "day_name"]).select_dtypes(include=np.number)
zero_var_cols = feature_only.columns[feature_only.nunique() <= 1].tolist()
print(f"Zero-variance (constant) feature columns: {len(zero_var_cols)}")
for c in zero_var_cols:
    print(f"  - {c}")

# Flow Duration == 0 artifacts
if "Flow Duration" in df.columns:
    n_zero_dur = int((df["Flow Duration"] == 0).sum())
    print(f"\nFlows with Flow Duration == 0 (CICFlowMeter artifact): {n_zero_dur:,}")

# Memory footprint BEFORE downcast (full float64/int64)
mem_before = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nDataFrame memory footprint (pre-downcast): {mem_before:,.1f} MB")

Zero-variance (constant) feature columns: 8
  - Bwd PSH Flags
  - Bwd URG Flags
  - Fwd Avg Bytes/Bulk
  - Fwd Avg Packets/Bulk
  - Fwd Avg Bulk Rate
  - Bwd Avg Bytes/Bulk
  - Bwd Avg Packets/Bulk
  - Bwd Avg Bulk Rate

Flows with Flow Duration == 0 (CICFlowMeter artifact): 0

DataFrame memory footprint (pre-downcast): 1,784.6 MB


In [9]:
# Safe numeric downcast (run AFTER all quality counts so float precision can't alter them).
# pd.to_numeric(downcast=...) chooses the smallest type that holds each column's range,
# so it never overflows (e.g. Destination Port max 65535 -> int32, not int16).
num_cols = df.select_dtypes(include=np.number).columns
ranges_before = df[num_cols].agg(["min", "max"])

for col in df.select_dtypes(include="integer").columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")
for col in df.select_dtypes(include="float").columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

# Verify the downcast preserved every column's value range (no overflow / no corruption).
ranges_after = df[num_cols].agg(["min", "max"])
assert np.allclose(ranges_before.to_numpy(), ranges_after.to_numpy(), rtol=1e-3, equal_nan=True), (
    "Downcast altered a column's min/max — aborting to avoid silent data corruption."
)

mem_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory footprint (post-downcast): {mem_after:,.1f} MB")
print(f"Reduction: {mem_before - mem_after:,.1f} MB ({(1 - mem_after / mem_before) * 100:.0f}% smaller)")
print(f"dtypes now: {df.select_dtypes(include=np.number).dtypes.value_counts().to_dict()}")

Memory footprint (post-downcast): 924.3 MB
Reduction: 860.3 MB (48% smaller)
dtypes now: {dtype('int32'): 27, dtype('int8'): 19, dtype('float64'): 15, dtype('float32'): 9, dtype('int16'): 7, dtype('int64'): 2}


### 1.6 — Data-quality summary

**Findings (all consistent with Engelen et al., 2021):**

- Column headers carried whitespace → stripped on load.
- `Flow Bytes/s` and `Flow Packets/s` carried `Inf` from divide-by-zero → converted to `NaN`; the
  affected rows were dropped (identical minimal cleaning for both strategies, so no leakage).
- A substantial fraction of rows are **exact duplicates** — the core leakage vector (Flaw 1). These
  are deliberately **kept** for now; the count is saved to `figures/dedup_stats.txt`.
- The label distribution is severely imbalanced (~80% BENIGN) with ultra-rare attack classes
  (Heartbleed, SQL Injection, Infiltration), foreshadowing Flaw 3.
- Zero-variance columns and the redundant `Fwd Header Length.1` are flagged for removal in Phase 3.

**Deferral note (deliberate):** Deduplication is *not* applied here. It is applied only inside
**Strategy B** so the difference between the duplicate-contaminated random split (A0) and the
deduplicated random split (A1) directly measures the F1 inflation caused by Flaw 1. This mirrors
the source paper's pipeline, which never mentions deduplication.

## Section 2 — Exploratory Data Analysis

EDA here is not decoration — every plot earns its place by motivating one of the three flaws or one
feature-engineering decision (PRD_eda §1):

- **Class distribution** → why accuracy / macro-F1 mislead (Flaw 3).
- **Attacks-by-day** → why a temporal split is the only honest evaluation (Flaw 2).
- **Inf/NaN audit** → the cleaning we applied in §1 (Flaw 1 context).
- **Distributions / skew** → justify the `log1p` transform (Phase 3 feature engineering).
- **Spearman correlation** → justify correlation-based feature pruning (ADR-1, ADR-8).

Plots that don't need all 2.8M rows are drawn on a fixed-seed 200k sample for speed (ADR-10);
class counts and shares are always computed on the full cleaned `df`.

### 2.1 — Class distribution (imbalance)

The dataset is dominated by BENIGN traffic, and the attack classes span five orders of magnitude in
frequency (DoS Hulk ~231k vs Heartbleed = 11). A log scale is required to see all 15 classes at
once. This extreme imbalance is the seed of **Flaw 3**: a model can ignore the rare classes
entirely and still post a high macro-average, because the rare classes contribute almost nothing to
the aggregate.

In [10]:
order = df["Label"].value_counts()
fig, ax = plt.subplots(figsize=(11, 6))
colors = ["#4C72B0" if lbl == "BENIGN" else "#C44E52" for lbl in order.index]
ax.bar(range(len(order)), order.values, color=colors)
ax.set_yscale("log")
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order.index, rotation=60, ha="right")
ax.set_ylabel("Flow count (log scale)")
ax.set_title("CICIDS2017 class distribution (log scale) — BENIGN (blue) vs 14 attack types (red)")
save_fig(fig, "01_class_distribution")

# Reuse the value_counts just computed — no extra full-frame boolean scan.
benign = order["BENIGN"] / order.sum() * 100
print(f"BENIGN: {benign:.1f}%  |  Attacks: {100 - benign:.1f}%")
print("Three rarest classes:", order.tail(3).to_dict())

Saved: ..\figures\01_class_distribution.png
BENIGN: 80.3%  |  Attacks: 19.7%
Three rarest classes: {'Infiltration': 36, 'Web Attack - Sql Injection': 21, 'Heartbleed': 11}


### 2.2 — Attack composition by weekday (motivates the temporal split)

Each attack family was executed on specific days of the capture week. A **random** split scatters
every day's attacks across both train and test, so the model trains on the very PortScan/DDoS
patterns it is later tested on — trivial leakage (**Flaw 2**). A **temporal** split (train Mon-Thu,
test Fri) mirrors a deployed IDS, which only ever sees *past* traffic. The stacked bar shows attack
volume per day; the printed table lists exactly which attack types appear on each day.

In [11]:
# Crosstab needs only two columns of df — no need to allocate a filtered attacks frame.
ct = (pd.crosstab(df["day_name"], df["Label"])
        .reindex(WEEKDAY_ORDER.keys())
        .drop(columns="BENIGN"))
fig, ax = plt.subplots(figsize=(12, 6))
ct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20", width=0.8)
ax.set_ylabel("Attack flow count")
ax.set_xlabel("")
ax.set_title("Attack flows by weekday (stacked) — each attack is confined to specific days")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=0)
save_fig(fig, "02_attacks_by_day")

print("Attack types present per day:")
for d in WEEKDAY_ORDER:
    present = sorted(ct.columns[ct.loc[d] > 0])
    print(f"  {d:<9}: {present if present else '(none)'}")

Saved: ..\figures\02_attacks_by_day.png
Attack types present per day:
  Monday   : (none)
  Tuesday  : ['FTP-Patator', 'SSH-Patator']
  Wednesday: ['DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'Heartbleed']
  Thursday : ['Infiltration', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']
  Friday   : ['Bot', 'DDoS', 'PortScan']


### 2.3 — Inf / missing-value audit

Reusing the pre-clean Inf counts captured in §1.3, the artifacts were confined to the two rate
features produced by division (`Flow Bytes/s`, `Flow Packets/s`) — exactly the divide-by-zero
pattern Engelen et al. (2021) describe. No other column carried Inf or NaN, which is why dropping
the affected rows cost only 0.1% of the data.

In [12]:
# inf_by_col was computed in section 1.3 (pre-clean Inf counts per column).
fig, ax = plt.subplots(figsize=(7, 3.5))
inf_by_col.sort_values().plot(kind="barh", ax=ax, color="#C44E52")
ax.set_xlabel("Inf cells (pre-clean)")
ax.set_title("Inf values were confined to two rate features (CICFlowMeter divide-by-zero)")
save_fig(fig, "03_missing_inf_heatmap")
print(inf_by_col.to_string())

Saved: ..\figures\03_missing_inf_heatmap.png
Flow Bytes/s      1509
Flow Packets/s    2867


### 2.4 — Distributions and skew (justifies log1p)

Network-flow features are heavily right-skewed and approximately power-law: most flows are tiny and
brief, with a long tail of large/long flows. The Flow Duration histogram (log x-axis) makes this
concrete. We then quantify skew across all numeric features and list those with |skew| > 3 — these
are the candidates for a `log1p` transform in Phase 3 (ADR-9).

In [13]:
sample = df.sample(n=min(200_000, len(df)), random_state=RANDOM_STATE)
fd_b = sample.loc[sample["Label"] == "BENIGN", "Flow Duration"].clip(lower=0)
fd_a = sample.loc[sample["Label"] != "BENIGN", "Flow Duration"].clip(lower=0)
top = max(fd_b.max(), fd_a.max()) + 1
bins = np.logspace(0, np.log10(top), 60)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(fd_b + 1, bins=bins, alpha=0.6, label="BENIGN", color="#4C72B0")
ax.hist(fd_a + 1, bins=bins, alpha=0.6, label="Attack", color="#C44E52")
ax.set_xscale("log")
ax.set_xlabel("Flow Duration (microseconds, log scale, +1 offset)")
ax.set_ylabel("Flow count")
ax.set_title("Flow Duration is heavily right-skewed \u2014 motivates log1p")
ax.legend()
save_fig(fig, "04_flow_duration_hist")

Saved: ..\figures\04_flow_duration_hist.png


In [14]:
# Drop tag columns AND the zero-variance columns before computing skew — a constant column
# has no meaningful skew, so excluding them keeps the count to real features only.
num_feats = df.drop(columns=["Label", "day", "day_name"] + zero_var_cols,
                    errors="ignore").select_dtypes(include=np.number)
skew = num_feats.skew().sort_values(ascending=False)
high_skew = skew[skew.abs() > 3]
print(f"Features with |skew| > 3: {len(high_skew)} of {len(skew)} numeric features")
print(high_skew.round(1).to_string())

Features with |skew| > 3: 52 of 70 numeric features
Total Length of Fwd Packets     805.2
Subflow Fwd Bytes               803.2
act_data_pkt_fwd                284.5
Subflow Bwd Packets             244.6
Total Backward Packets          244.6
Total Fwd Packets               244.3
Subflow Fwd Packets             244.3
Subflow Bwd Bytes               244.2
Total Length of Bwd Packets     244.2
Fwd URG Flags                    94.7
CWE Flag Count                   94.7
RST Flag Count                   64.2
ECE Flag Count                   64.0
Active Min                       47.7
Flow Bytes/s                     46.4
Active Std                       40.5
Active Mean                      38.2
Active Max                       24.3
Flow IAT Min                     23.8
Bwd Packets/s                    21.5
Fwd Packet Length Min            20.1
Down/Up Ratio                    12.0
Fwd Packet Length Std            10.5
Idle Std                         10.5
Min Packet Length                10.

### 2.5 — Packet-length outliers

Boxplots of forward/backward packet-length features (drawn on the sample, symlog y-axis) show the
extreme right tails and outliers typical of mixed traffic. This reinforces the choice of **Spearman**
(rank-based, outlier-robust) over Pearson for the correlation analysis that follows.

In [15]:
pkt_cols = [c for c in ["Fwd Packet Length Max", "Fwd Packet Length Mean",
                        "Bwd Packet Length Max", "Bwd Packet Length Mean"] if c in df.columns]
fig, ax = plt.subplots(figsize=(10, 5))
sample[pkt_cols].clip(lower=0).plot(kind="box", ax=ax)
ax.set_yscale("symlog")
ax.set_ylabel("Bytes (symlog)")
ax.set_title("Packet-length features: heavy right tails and outliers")
plt.xticks(rotation=20, ha="right")
save_fig(fig, "05_packet_length_box")

Saved: ..\figures\05_packet_length_box.png


### 2.6 — Spearman correlation and feature redundancy (ADR-1, ADR-8)

**We use Spearman rank correlation, not Pearson, because:**

1. **Non-normality.** Flow features (packet lengths, byte counts, durations, IATs) are heavily
   right-skewed and approximately power-law; Pearson's normality/linearity assumptions fail.
2. **Outlier robustness.** Real traffic has extreme flows (large transfers, scans). Spearman uses
   ranks, so a few extremes do not dominate the coefficient.
3. **Monotonic, not linear.** For redundancy we only care whether two features rise/fall together,
   which is exactly what rank correlation captures.
4. **Cost.** Kendall τ shares the robustness but is ~O(n²); Spearman is O(n log n) (a sort) and
   scales to millions of rows.

Below we visualise the top-20 variance features for legibility. The **actual** redundancy pruning
(Phase 3, §3.2) recomputes Spearman across *all* candidate features at |r| > 0.95 — this heatmap is
illustrative, not the pruning basis.

In [16]:
# Reuse the 200k `sample` from §2.4 (identical rows) and compute the ranking variance on it too
# — no second df.sample() and no full-frame variance scan.
sample_feats = sample.drop(columns=["Label", "day", "day_name"] + zero_var_cols,
                           errors="ignore").select_dtypes(include=np.number)
top_var = sample_feats.var().sort_values(ascending=False).head(20).index.tolist()
spearman = sample[top_var].corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(spearman, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Spearman correlation — top-20 variance features")
save_fig(fig, "06_spearman_heatmap")

cols = spearman.columns
pairs = [(cols[i], cols[j], round(float(spearman.iloc[i, j]), 3))
         for i in range(len(cols)) for j in range(i + 1, len(cols))
         if abs(spearman.iloc[i, j]) > 0.95]
print(f"Feature pairs with |Spearman r| > 0.95: {len(pairs)}")
for a, b, r in sorted(pairs, key=lambda t: -abs(t[2])):
    print(f"  {r:+.3f}  {a}  <->  {b}")

Saved: ..\figures\06_spearman_heatmap.png
Feature pairs with |Spearman r| > 0.95: 12
  +1.000  Idle Max  <->  Idle Mean
  +0.999  Idle Mean  <->  Idle Min
  +0.999  Bwd IAT Max  <->  Bwd IAT Mean
  +0.998  Bwd IAT Total  <->  Bwd IAT Max
  +0.998  Idle Max  <->  Idle Min
  +0.997  Bwd IAT Total  <->  Bwd IAT Mean
  +0.997  Fwd IAT Max  <->  Fwd IAT Mean
  +0.995  Fwd IAT Total  <->  Fwd IAT Max
  +0.992  Fwd IAT Total  <->  Fwd IAT Mean
  +0.991  Flow Duration  <->  Flow IAT Max
  +0.980  Flow IAT Max  <->  Flow IAT Mean
  +0.969  Flow Duration  <->  Flow IAT Mean


### 2.7 — EDA takeaways (feeding feature engineering and the critique)

- **Imbalance (Flaw 3 seed).** BENIGN is 80.3% of flows; Heartbleed (11), SQL Injection (21) and
  Infiltration (36) are too small to move accuracy or macro-F1. Any aggregate-only score — like the
  source paper's macro F1 — can therefore hide total failure on these classes. Note the source
  reports **no per-class metrics and no imbalance handling**; we will (Phase 6).
- **Real-world framing.** Production networks are *far* more benign-heavy than 80%, yet even at this
  comparatively attack-rich ratio models still collapse on rare attacks — the problem is the class
  rarity, not just the overall ratio.
- **Temporal structure (Flaw 2).** Attacks are confined to specific days, so a random split leaks
  attack signatures into training. Strategy B trains on Mon-Thu and tests on Friday (ADR-7).
- **Skew (feature engineering).** The flagged |skew| > 3 features will receive a `log1p` transform
  (ADR-9); Spearman confirms the redundancy clusters that correlation pruning at |r| > 0.95 will
  remove (ADR-8), shrinking the ~70 usable features before modelling.

## Section 3 — Feature Engineering

With the data loaded, cleaned, and profiled, we now build the **model-ready feature set**. Every
transformation here is applied *uniformly* to all runs so that the only intentional differences
between Strategy A (source-style) and Strategy B (honest) remain the three flaws under study —
deduplication (Flaw 1), the temporal split (Flaw 2), and per-class metrics (Flaw 3). Feature
engineering is deliberately **not** a fourth confound: structural and redundancy drops remove only
non-informative or duplicated columns, and `log1p` is monotonic (so it cannot change tree-model
splits). Scaling and SMOTE are *not* done here — they are fit per split inside Phase 5 to prevent
leakage (INV-2).


### 3.1 — Structural column removal

Two groups of columns carry no usable signal and are dropped for **all** strategies:

1. **`Fwd Header Length.1`** — an exact duplicate of `Fwd Header Length` (a known CICFlowMeter
   export quirk noted by Engelen et al., 2021).
2. **Zero-variance constants** — the columns flagged in §1.5; a constant feature cannot help any
   model and only inflates the feature count.


In [17]:
# Idempotent guard: re-strip column names (already done in §1.1; safe to repeat).
df.columns = df.columns.str.strip()

NON_FEATURE = ["Label", "day", "day_name"]

# Redundant duplicate column (CICFlowMeter quirk; Engelen et al., 2021).
redundant_cols = [c for c in ["Fwd Header Length.1"] if c in df.columns]

# Structural drops apply to every strategy (they remove junk, not flaw-relevant signal).
drop_structural = sorted(set(redundant_cols) | set(zero_var_cols))

candidate_features = [c for c in df.columns if c not in NON_FEATURE]
struct_cols = [c for c in candidate_features if c not in drop_structural]

print(f"Candidate features (pre-engineering): {len(candidate_features)}")
print(f"  - redundant duplicate columns dropped : {redundant_cols}")
print(f"  - zero-variance columns dropped       : {len(zero_var_cols)}")
print(f"Surviving after structural drops        : {len(struct_cols)}")

Candidate features (pre-engineering): 78
  - redundant duplicate columns dropped : ['Fwd Header Length.1']
  - zero-variance columns dropped       : 8
Surviving after structural drops        : 69


### 3.2 — Correlation pruning (ADR-8)

Near-redundant features add collinearity without new information. Following ADR-8 we prune at
**|Spearman r| > 0.95**, computed here over **all** candidate features — the §2.6 heatmap was
restricted to the top-20 variance features purely for legibility, but redundancy must be assessed
across the full set. We rank features by variance and greedily keep the highest-variance
representative of each correlated cluster, dropping the rest. Spearman is rank-based (robust to the
heavy skew/outliers in flow data) and cheap on the 200k sample; this is more transparent and stable
than VIF on near-collinear flow features.

In [18]:
# Correlation pruning over ALL candidate features (not just the top-20 EDA subset).
# The §2.6 heatmap used the top-20 variance features for readability; redundancy, however,
# must be checked across every candidate, so we recompute the Spearman matrix on all
# struct_cols here (on the same 200k sample — rank-based, O(n log n), cheap).
struct_sample = sample[struct_cols]
var_rank = struct_sample.var().sort_values(ascending=False)
corr = struct_sample.corr(method="spearman").abs()

# Greedy: walk features high -> low variance; drop any feature that is |r| > 0.95 with a
# higher-variance feature already kept. Keeps one representative per redundant cluster.
kept, corr_drop = [], []
for col in var_rank.index:
    if any(corr.loc[col, k] > 0.95 for k in kept):
        corr_drop.append(col)
    else:
        kept.append(col)
corr_drop = sorted(corr_drop)
feature_cols = [c for c in struct_cols if c not in corr_drop]

n_pairs = int((np.triu(corr.to_numpy(), k=1) > 0.95).sum())
print(f"Candidate features checked: {len(struct_cols)}  |  redundant pairs (|r|>0.95): {n_pairs}")
print(f"Correlation-pruned features ({len(corr_drop)}):")
for c in corr_drop:
    print(f"  - {c}")
print(f"\nFinal engineered feature count: {len(feature_cols)} "
      f"(from {len(candidate_features)} candidates)")

Candidate features checked: 69  |  redundant pairs (|r|>0.95): 57
Correlation-pruned features (29):
  - Active Max
  - Active Mean
  - Active Min
  - Avg Bwd Segment Size
  - Bwd IAT Max
  - Bwd IAT Mean
  - Bwd Packet Length Max
  - Bwd Packet Length Mean
  - CWE Flag Count
  - Flow IAT Max
  - Flow IAT Mean
  - Flow Packets/s
  - Fwd IAT Max
  - Fwd IAT Mean
  - Fwd Packet Length Mean
  - Fwd Packets/s
  - Idle Mean
  - Idle Min
  - Max Packet Length
  - Min Packet Length
  - Packet Length Mean
  - Packet Length Std
  - RST Flag Count
  - SYN Flag Count
  - Subflow Bwd Packets
  - Subflow Fwd Bytes
  - Subflow Fwd Packets
  - Total Backward Packets
  - Total Length of Bwd Packets

Final engineered feature count: 40 (from 78 candidates)


### 3.3 — Label construction (binary + multiclass)

We build **both** label granularities (ADR-4): a binary target drives the headline F1 comparison,
while the multiclass target is required to expose the rare-class recall collapse (Flaw 3) that an
aggregate-only metric hides. Labels are kept separate from the feature matrix.


In [19]:
from sklearn.preprocessing import LabelEncoder  # preprocessing utility, not a model

# Binary target: BENIGN (0) vs any attack (1).
y_bin = (df["Label"] != "BENIGN").astype("int8")

# Multiclass target: needed to surface per-class failure (Flaw 3).
label_encoder = LabelEncoder()
y_multi = label_encoder.fit_transform(df["Label"])
label_names = list(label_encoder.classes_)

print(f"Binary  - attack rate: {y_bin.mean() * 100:.2f}%  "
      f"({int(y_bin.sum()):,} attacks / {len(y_bin):,} flows)")
print(f"Multiclass - {len(label_names)} classes:")
for cls, n in df["Label"].value_counts().items():
    print(f"  {cls:34s} {n:>9,}")

Binary  - attack rate: 19.68%  (556,556 attacks / 2,827,876 flows)
Multiclass - 15 classes:
  BENIGN                             2,271,320
  DoS Hulk                             230,124
  PortScan                             158,804
  DDoS                                 128,025
  DoS GoldenEye                         10,293
  FTP-Patator                            7,935
  SSH-Patator                            5,897
  DoS slowloris                          5,796
  DoS Slowhttptest                       5,499
  Bot                                    1,956
  Web Attack - Brute Force               1,507
  Web Attack - XSS                         652
  Infiltration                              36
  Web Attack - Sql Injection                21
  Heartbleed                                11


### 3.4 — Feature matrix

`X` contains **only** engineered traffic features. The `day` / `day_name` columns are *not*
features (INV-4): they encode the temporal split, not traffic behaviour, so feeding them to a model
would leak the very signal Strategy B is meant to test. We keep `df['day']` aside as the split key
for Phase 5. Duplicate rows are still present **by design** — removing them is a Strategy-B step, so
runs A0 (with duplicates) vs A1 (deduplicated) can isolate Flaw 1.


In [20]:
X = df[feature_cols].copy()
day = df["day"].copy()  # split key only — never fed to a model

print(f"X shape: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Duplicate rows still present in X (by design): {int(X.duplicated().sum()):,}")
assert "Label" not in X.columns
assert "day" not in X.columns and "day_name" not in X.columns

X shape: 2,827,876 rows x 40 features


Duplicate rows still present in X (by design): 308,708


### 3.5 — Log-transform of skewed features (ADR-9)

Network-flow features are heavily right-skewed (§2.4). For the features with **|skew| > 3** we apply
`log1p`, *guarding* on the column minimum because `log1p` requires non-negative input and §1 flagged
CICFlowMeter columns with spurious negatives — those are left untransformed and reported.

`log1p` is **monotonic**, so it does not change the split points of tree models (RF, XGBoost,
Isolation Forest are all invariant to it); we apply it for distribution sanity and any
scale-sensitive analysis. Crucially, it has **no fitted parameters**, so it cannot leak and is safe
to apply per split in Phase 5. We therefore define a reusable `apply_log1p()` rather than baking the
transform into a stored matrix.


In [21]:
col_min = X.min()
skewed_features = [c for c in feature_cols if c in high_skew.index and col_min[c] >= 0]
excluded_neg = [c for c in feature_cols if c in high_skew.index and col_min[c] < 0]

n_skewed_surviving = sum(c in high_skew.index for c in feature_cols)
print(f"Skewed features (|skew|>3) surviving pruning: {n_skewed_surviving}")
print(f"  -> log1p applied to {len(skewed_features)} non-negative features")
print(f"  -> {len(excluded_neg)} skewed-but-negative features left untransformed: {excluded_neg}")


def apply_log1p(frame, cols):
    """Return a copy of `frame` with log1p applied to `cols`.

    `cols` is an explicit required argument (no global capture), so the function is pure and
    safe to run out-of-order. Deterministic and stateless (no fitted parameters) -> cannot leak;
    applied per split in Phase 5. Monotonic -> does not alter tree-model splits.
    """
    out = frame.copy()
    out[cols] = np.log1p(out[cols])
    return out


print("apply_log1p(frame, cols) defined - call as apply_log1p(X_train, skewed_features) in Phase 5.")

Skewed features (|skew|>3) surviving pruning: 30
  -> log1p applied to 23 non-negative features
  -> 7 skewed-but-negative features left untransformed: ['Flow Bytes/s', 'Flow IAT Min', 'Fwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Init_Win_bytes_backward', 'min_seg_size_forward']
apply_log1p(frame, cols) defined - call as apply_log1p(X_train, skewed_features) in Phase 5.


### 3.6 — Leakage guard

A quick automated check that no target-derived or split-encoding column slipped into the feature
set. This is cheap insurance against the exact class of mistake (leakage) that the whole project
critiques.


In [22]:
leak_terms = ("label", "day")
suspicious = [c for c in feature_cols if any(t in c.lower() for t in leak_terms)]
assert not suspicious, f"Potential leakage columns in X: {suspicious}"
assert "Label" not in feature_cols
assert all(c not in feature_cols for c in ["day", "day_name"])
print(f"Leakage check passed: X holds only traffic features ({len(feature_cols)} cols); "
      "labels and split keys are held separately.")

Leakage check passed: X holds only traffic features (40 cols); labels and split keys are held separately.


### 3.7 — Feature-engineering ledger

Every dropped column is accounted for (INV-6 — auditable cleaning). The engineered frame is also
persisted to a git-ignored parquet so Phase 5 can reload it without re-running §1–§3.


In [23]:
ledger = pd.DataFrame(
    [
        ("raw feature columns", len(candidate_features)),
        ("- redundant duplicate (Fwd Header Length.1)", -len(redundant_cols)),
        ("- zero-variance constants", -len(zero_var_cols)),
        ("- correlation-redundant (|Spearman r|>0.95)", -len(corr_drop)),
        ("= final engineered features", len(feature_cols)),
        ("  (of which log1p-transformed: skew>3, >=0)", len(skewed_features)),
    ],
    columns=["step", "delta"],
)
print(ledger.to_string(index=False))

# (P2) Persist engineered frame for fast Phase 5 re-runs. data/ is git-ignored.
# Slice df directly (no second full copy of X); store the raw Label + split key so Phase 5
# rebuilds y_bin/y_multi cheaply on load.
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)
parquet_path = PROCESSED_DIR / "engineered.parquet"
try:
    df[feature_cols + ["day", "Label"]].to_parquet(parquet_path, index=False)
    print(f"\nSaved engineered frame -> data/processed/engineered.parquet "
          f"({len(df):,} x {len(feature_cols) + 2})")
except Exception as exc:  # pyarrow/fastparquet may be absent — non-fatal (P2)
    print(f"\nParquet persist skipped (no parquet engine available): {exc}")

                                       step  delta
                        raw feature columns     78
- redundant duplicate (Fwd Header Length.1)     -1
                  - zero-variance constants     -8
- correlation-redundant (|Spearman r|>0.95)    -29
                = final engineered features     40
  (of which log1p-transformed: skew>3, >=0)     23



Saved engineered frame -> data/processed/engineered.parquet (2,827,876 x 42)
